<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Data-Quality-Validation-Framework/blob/main/Enterprise_Data_Quality_Validation_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import random
import json

import pandas as pd
import numpy as np

BASE_PATH = Path('/content/enterprise_data_quality_validation_framework')
SOURCE_PATH = BASE_PATH / 'data' / 'source'
TARGET_PATH = BASE_PATH / 'data' / 'target'
OUTPUT_PATH = BASE_PATH / 'outputs'
RULES_PATH = BASE_PATH / 'rules'

for p in [SOURCE_PATH, TARGET_PATH, OUTPUT_PATH, RULES_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/enterprise_data_quality_validation_framework


In [2]:
random.seed(42)
np.random.seed(42)

num_records = 5000
regions = ['North', 'South', 'East', 'West', 'Central']
start_date = datetime(2024, 1, 1)

source_data = []
for i in range(1, num_records + 1):
    event_date = start_date + timedelta(days=random.randint(0, 30))
    source_data.append({
        'record_id': f'REC{i:06d}',
        'customer_id': f'CUST{random.randint(1, 1500):05d}',
        'region': random.choice(regions),
        'event_date': event_date.strftime('%Y-%m-%d'),
        'usage_value': round(float(max(1, np.random.normal(35, 10))), 2),
        'load_timestamp': (event_date + timedelta(hours=2)).strftime('%Y-%m-%d %H:%M:%S')
    })

source_df = pd.DataFrame(source_data)
source_df.to_csv(SOURCE_PATH / 'source_usage_data.csv', index=False)

# Create target dataset and intentionally inject some quality issues

target_df = source_df.copy()

target_df.loc[5, 'usage_value'] = None
target_df.loc[10, 'customer_id'] = None
target_df = pd.concat([target_df, target_df.iloc[[0]]], ignore_index=True)
target_df = target_df.drop(index=[20, 21, 22])
target_df.loc[0, 'load_timestamp'] = '2023-01-01 00:00:00'  # stale/freshness issue example

target_df.to_csv(TARGET_PATH / 'target_usage_data.csv', index=False)

print('Source and target datasets created successfully.')
print(SOURCE_PATH / 'source_usage_data.csv')
print(TARGET_PATH / 'target_usage_data.csv')

Source and target datasets created successfully.
/content/enterprise_data_quality_validation_framework/data/source/source_usage_data.csv
/content/enterprise_data_quality_validation_framework/data/target/target_usage_data.csv


In [3]:
quality_rules = {
    'dataset_name': 'usage_data',
    'schema_required_columns': ['record_id', 'customer_id', 'region', 'event_date', 'usage_value', 'load_timestamp'],
    'freshness_column': 'load_timestamp',
    'freshness_max_age_days': 60,
    'duplicate_key_columns': ['record_id'],
    'null_check_columns': ['record_id', 'customer_id', 'region', 'event_date', 'usage_value'],
    'reconciliation_checks': ['row_count', 'sum_usage_value']
}

with open(RULES_PATH / 'data_quality_rules.json', 'w') as f:
    json.dump(quality_rules, f, indent=2)

print(json.dumps(quality_rules, indent=2))

{
  "dataset_name": "usage_data",
  "schema_required_columns": [
    "record_id",
    "customer_id",
    "region",
    "event_date",
    "usage_value",
    "load_timestamp"
  ],
  "freshness_column": "load_timestamp",
  "freshness_max_age_days": 60,
  "duplicate_key_columns": [
    "record_id"
  ],
  "null_check_columns": [
    "record_id",
    "customer_id",
    "region",
    "event_date",
    "usage_value"
  ],
  "reconciliation_checks": [
    "row_count",
    "sum_usage_value"
  ]
}


In [4]:
source_df = pd.read_csv(SOURCE_PATH / 'source_usage_data.csv')
target_df = pd.read_csv(TARGET_PATH / 'target_usage_data.csv')

with open(RULES_PATH / 'data_quality_rules.json', 'r') as f:
    rules = json.load(f)

print('Datasets and rules loaded successfully.')
print('Source rows:', len(source_df))
print('Target rows:', len(target_df))

Datasets and rules loaded successfully.
Source rows: 5000
Target rows: 4998


In [5]:
quality_results = []

def add_result(check_name, status, issue_count, details):
    quality_results.append({
        'check_name': check_name,
        'status': status,
        'issue_count': issue_count,
        'details': details,
        'timestamp': datetime.now().isoformat()
    })

required_columns = rules['schema_required_columns']
missing_columns = [col for col in required_columns if col not in target_df.columns]

if missing_columns:
    add_result('schema_validation', 'FAILED', len(missing_columns), f'Missing columns: {missing_columns}')
else:
    add_result('schema_validation', 'PASSED', 0, 'All required columns are present')

In [6]:
target_df['load_timestamp'] = pd.to_datetime(target_df['load_timestamp'], errors='coerce')
max_age_days = rules['freshness_max_age_days']
threshold_date = datetime.now() - timedelta(days=max_age_days)
stale_count = int((target_df['load_timestamp'] < threshold_date).sum())

if stale_count > 0:
    add_result('freshness_validation', 'FAILED', stale_count, f'{stale_count} records are older than allowed threshold')
else:
    add_result('freshness_validation', 'PASSED', 0, 'All records are within freshness threshold')

In [7]:
duplicate_keys = rules['duplicate_key_columns']
duplicate_count = int(target_df.duplicated(subset=duplicate_keys).sum())

if duplicate_count > 0:
    add_result('duplicate_detection', 'FAILED', duplicate_count, f'{duplicate_count} duplicate records found')
else:
    add_result('duplicate_detection', 'PASSED', 0, 'No duplicate records found')

In [8]:
null_columns = rules['null_check_columns']
null_issue_count = 0
null_details = []

for col in null_columns:
    count = int(target_df[col].isna().sum())
    if count > 0:
        null_issue_count += count
        null_details.append(f'{col}: {count}')

if null_issue_count > 0:
    add_result('null_handling_check', 'FAILED', null_issue_count, '; '.join(null_details))
else:
    add_result('null_handling_check', 'PASSED', 0, 'No null issues found in required columns')

In [9]:
reconciliation_results = []

source_row_count = len(source_df)
target_row_count = len(target_df)
row_count_diff = source_row_count - target_row_count

reconciliation_results.append({
    'check_name': 'row_count_reconciliation',
    'source_value': source_row_count,
    'target_value': target_row_count,
    'difference': row_count_diff,
    'status': 'PASSED' if row_count_diff == 0 else 'FAILED'
})

source_sum_usage = round(source_df['usage_value'].sum(), 2)
target_sum_usage = round(target_df['usage_value'].fillna(0).sum(), 2)
sum_diff = round(source_sum_usage - target_sum_usage, 2)

reconciliation_results.append({
    'check_name': 'sum_usage_value_reconciliation',
    'source_value': source_sum_usage,
    'target_value': target_sum_usage,
    'difference': sum_diff,
    'status': 'PASSED' if sum_diff == 0 else 'FAILED'
})

if row_count_diff == 0 and sum_diff == 0:
    add_result('reconciliation_check', 'PASSED', 0, 'Source and target reconciliation matched')
else:
    add_result('reconciliation_check', 'FAILED', 1, 'Source and target reconciliation differences found')

In [10]:
quality_report_df = pd.DataFrame(quality_results)
reconciliation_report_df = pd.DataFrame(reconciliation_results)

quality_report_df.to_csv(OUTPUT_PATH / 'quality_report.csv', index=False)
reconciliation_report_df.to_csv(OUTPUT_PATH / 'reconciliation_report.csv', index=False)

quality_report_df

,check_name,status,issue_count,details,timestamp
0,schema_validation,PASSED,0,All required columns are present,2026-03-17T05:11:37.232798
1,freshness_validation,FAILED,4998,4998 records are older than allowed threshold,2026-03-17T05:12:00.310699
2,duplicate_detection,FAILED,1,1 duplicate records found,2026-03-17T05:12:32.014969
3,null_handling_check,FAILED,2,customer_id: 1; usage_value: 1,2026-03-17T05:12:56.355489
4,reconciliation_check,FAILED,1,Source and target reconciliation differences f...,2026-03-17T05:13:18.904888


In [11]:
summary = {
    'generated_at': datetime.now().isoformat(),
    'total_checks': len(quality_report_df),
    'passed_checks': int((quality_report_df['status'] == 'PASSED').sum()),
    'failed_checks': int((quality_report_df['status'] == 'FAILED').sum()),
    'source_rows': source_row_count,
    'target_rows': target_row_count
}

with open(OUTPUT_PATH / 'quality_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "generated_at": "2026-03-17T05:14:26.207867",
  "total_checks": 5,
  "passed_checks": 1,
  "failed_checks": 4,
  "source_rows": 5000,
  "target_rows": 4998
}


In [12]:
reconciliation_report_df

,check_name,source_value,target_value,difference,status
0,row_count_reconciliation,5000.00,4998.00,2.00,FAILED
1,sum_usage_value_reconciliation,175279.84,175169.07,110.77,FAILED
